# Deep agent: subgraphs as tools

Worktree-local notebook. The compiled graphs stay as they are; this notebook
wires them as LangChain tools and optionally asks a deep agent to choose
among them.

Run Jupyter from this worktree root after `uv sync`.

```text
user request
     ↓
create_deep_agent
     ↓
cohort / classify_* / discover_* / recommend_pathway / persist_recc
     ↓
compiled subgraphs + TaskStore
```

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from dotenv import load_dotenv

print(load_dotenv(".env", override=True))
key = os.getenv("OPENAI_API_KEY")
print(key[:10] if key else "OPENAI_API_KEY not found")

from playbook import (
    SYSTEM_PROMPT,
    classify_intent,
    classify_subflow,
    cohort,
    configure_runtime,
    create_playbook_agent,
    discover_intent,
    discover_subflow,
    persist_recc,
    recommend_pathway,
)
from playbook import runtime as rt
from playbook.fill import EDA_DATA_DIR

ROOT = Path(".").resolve()
playbook, store = configure_runtime(
    data_dir=EDA_DATA_DIR,
    store_path=EDA_DATA_DIR / "run_store.sqlite",
)



print("tasks", len(store.fetchall("SELECT task_id FROM tasks")))
print("intents", playbook.intent_ids())
print("LANGSMITH_TRACING", os.environ.get("LANGSMITH_TRACING"))

True
sk-svcacct
tasks 145
intents ['account_access', 'order_issue']
LANGSMITH_TRACING true


## Direct tool calls

These invoke the compiled graphs with no model. Use this cell to confirm
the SQLite store survives LangGraph worker threads.

In [2]:
cohort_out = cohort.invoke(
    {
        "start_date": "2026-09-01",
        "end_date": "2026-09-07",
        "method": "jaccard",
        "cohort_query": {},
        "run_id": "notebook-deep-agent",
    }
)
intent_out = classify_intent.invoke({})
subflow_out = classify_subflow.invoke({})
print(cohort_out)
print(intent_out)
print(subflow_out)

{'run_id': 'notebook-deep-agent', 'current_stage': 'summarize_cohort', 'cohort_summary': {'start': '2026-09-01', 'end': '2026-09-07', 'n': 51, 'min_conversation_date': '2026-09-01', 'max_conversation_date': '2026-09-07', 'method': 'jaccard'}, 'kb_version': 1}
{'current_stage': 'summarize_intent_assignments', 'intent_summary': {'total': 51, 'processed': 51, 'classified': 15, 'unknown': 36, 'low_confidence': 10, 'by_intent': {'unknown': 36, 'order_issue': 15}, 'still_unresolved': 36}, 'kb_version': 1}
{'current_stage': 'summarize_subflow_assignments', 'subflow_summary': {'processed': 15, 'classified': 0, 'unknown': 15, 'low_confidence': 0, 'by_intent': {'order_issue': {'unknown': 15}}, 'still_unresolved': 15}, 'kb_version': 1}


Discovery and pathway tools. `recommend_pathway` drafts only;
`persist_recc` writes the store / KB.

In [6]:
print("------------\ndiscover_intent")
print(discover_intent.invoke({}))
print("\n------------\ndiscover_subflow")
print(discover_subflow.invoke({}))
print("\n------------\nrecommend_pathway")
print(recommend_pathway.invoke({"target_subflow": "reset_2fa"}))
print("\n------------\persist_recc_pathway")
print(persist_recc.invoke({"target_subflow": "reset_2fa"}))
print("\n------------\nlist reccs")
print("recs", store.list_recommendations(rt.current_run_id))
print("\n------------\nkb version")
print("kb_version", playbook.version)

------------
discover_intent
{'current_stage': 'summarize_intent_discovery', 'discovery_summary': {'intents': {'unresolved_tasks': 2, 'candidate_count': 0, 'outlier_count': 0, 'approved_count': 0, 'rejected_count': 0}}, 'pending_proposal_ids': [], 'approved_change_ids': [], 'kb_version': 2}

------------
discover_subflow
{'current_stage': 'summarize_subflow_discovery', 'discovery_summary': {'subflows': {'unresolved_tasks': 49, 'candidate_count': 0, 'emerging_count': 2, 'approved_count': 0, 'rejected_count': 2}}, 'pending_proposal_ids': ['prop_41068d46', 'prop_ae102c1e'], 'approved_change_ids': [], 'kb_version': 2}

------------
recommend_pathway
{'current_stage': 'summarize_pathway', 'target_subflow': 'reset_2fa', 'recommendation_summary': {'items': [{'subflow_id': 'reset_2fa', 'intent_id': None, 'supported': False, 'n_tasks': 0, 'support': 0.0}], 'recommended': 0}, 'evaluation': {'supported': False, 'successful': 0, 'support': 0.0}, 'kb_version': 2}

------------\persist_recc_pathway


### current seeded data
seed: ~Aug 25–Sep 8

noise: Sep 9–10

status_payment_method: Sep 10–12

slow_speed: Sep 12–14

## Deep agent (optional)

Needs `OPENAI_API_KEY`. The agent picks tools from the request instead of
following a fixed classify → discover → recommend graph.

In [7]:
from langchain.chat_models import init_chat_model

if os.environ.get("OPENAI_API_KEY"):
    model = init_chat_model("openai:gpt-4.1-mini", temperature=0)
    agent = create_playbook_agent(model=model)
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Classify the 2026-09-01 to 2026-09-07 cohort with jaccard. "
                        "Stop after classification unless unresolved tasks clearly "
                        "need discovery."
                    ),
                }
            ]
        },
        config={"configurable": {"thread_id": "notebook-deep-agent"}},
    )
    print(result["messages"][-1].content)
    result["messages"][-1].content.pretty_print()
    
else:
    print("Set OPENAI_API_KEY to invoke create_playbook_agent.")
    print("System prompt starts with:")
    print(SYSTEM_PROMPT.splitlines()[0])

The classification of the cohort from 2026-09-01 to 2026-09-07 with the jaccard method resulted in 51 tasks processed. Out of these, 15 tasks were classified under the existing intent "order_issue," but 36 tasks remain unknown and unresolved. Since a significant number of tasks remain unresolved, discovery of new intents may be warranted. Would you like me to proceed with discovering new intents among the unresolved tasks?


In [12]:
thread_id="notebook-deep-agent"
def user_message(agent, content, thread_id):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        content
                    ),
                }
            ]
        },
        config={"configurable": {"thread_id": thread_id}},
    )
    #print(result["messages"][-1].content)
    result["messages"][-1].pretty_print()

In [9]:
content = """yes, run discovery over this cohort"""
user_message(agent, content, thread_id)

Discovery over the cohort from 2026-09-01 to 2026-09-07 identified one candidate new intent among the unresolved tasks. This new intent has been approved for inclusion in the knowledge base.

If you would like, I can now proceed to classify the unresolved tasks with this new intent and update the knowledge base accordingly. Would you like me to do that?


AttributeError: 'str' object has no attribute 'pretty_print'

In [11]:
content = """what is the name of the new intent?"""
user_message(agent, content, thread_id)

I could not find the name of the new intent from the discovery proposal ID in the knowledge base files. It appears the new intent is approved but the name is not directly accessible from the current files.

I can provide a summary of the discovered intent from the discovery process or attempt to retrieve the candidate intent name from the discovery metadata if you want. How would you like to proceed?
================================== Ai Message ==================================
Name: playbook_deep_agent

I could not find the name of the new intent from the discovery proposal ID in the knowledge base files. It appears the new intent is approved but the name is not directly accessible from the current files.

I can provide a summary of the discovered intent from the discovery process or attempt to retrieve the candidate intent name from the discovery metadata if you want. How would you like to proceed?


In [13]:
content = """what are the available intent labels in the kb?"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

It appears that the knowledge base files are currently empty or not accessible, so I cannot retrieve the list of available intent labels from the knowledge base files directly.

However, from the classification summary, I can tell you that one existing intent label is "order_issue." Additionally, a new intent has been approved from the discovery process, but its name is not currently accessible.

Would you like me to proceed with classifying the unresolved tasks using the new intent or assist you with something else?


In [14]:
content = """yes, classify unresolved tasks with the new intent"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The unresolved tasks from the cohort have been classified with the new intent "shipping_issue." Now, 34 tasks are classified under "shipping_issue," 15 under "order_issue," and only 2 tasks remain unknown.

Is there anything else you would like to do with this cohort or the knowledge base?


In [15]:
content = """classify empty subflows"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

All tasks with empty subflows in the cohort from 2026-09-01 to 2026-09-07 remain unclassified for subflows. There are 49 such tasks, with 34 under the "shipping_issue" intent and 15 under the "order_issue" intent.

Would you like me to proceed with discovering new subflows for either or both of these intents?


In [16]:
content = """yes, for both"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

Discovery for new subflows in both the "shipping_issue" and "order_issue" intents found no candidate subflows but identified 2 emerging subflows in each intent. However, none of these emerging subflows have been approved yet.

Would you like me to review the emerging subflows for either intent or take any other action?


In [17]:
content = """yes, review the emerging subflows"""
user_message(agent, content, thread_id)

================================== Ai Message ==================================
Name: playbook_deep_agent

The review of the emerging subflows for both the "shipping_issue" and "order_issue" intents shows that none of the emerging subflows have supporting successful task traces. There is no evidence to recommend these subflows for inclusion in the knowledge base at this time.

Would you like to explore other actions or focus on a different aspect of the knowledge base?
